## [A] Importing & Preparing Raw Data

The goal of this Jupyter notebook is to collect all the different data from all sources and save it into one xlsx file in a usable format.

In [1]:
import pandas as pd
import numpy as np
import os
from pandas.tseries.offsets import MonthEnd, QuarterEnd # for end of month date adjustment

In [ ]:
# Sets pandas display option to format floating-point numbers to six decimal places
pd.set_option('display.float_format', '{:.6f}'.format)

## 1. Import Raw Refinitiv Data

I first import all the data that I downloaded from Refinitiv & Other resources. 

In [ ]:
path = "data/input_data/"
failed_path = "data/input_data/failed/"

# Sets path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [4]:
# Refinitiv Data
constituents_presence_matrix_file = pd.read_excel("data/input_data/constituents__STOXX.xlsx", sheet_name="Presence_Matrix") # Refinitiv Data
constituents_name_list_file = pd.read_excel("data/input_data/constituents__STOXX.xlsx", sheet_name="Constituents") # Refinitiv Data

daily_history_file = pd.read_csv(path + "daily_stock_history_file_data_20260309_094022.csv") # Refinitiv Data
daily_index_history_file = pd.read_csv(path + "daily_index_history_file_data_20260309_094107.csv") # Refinitiv Data

quarterly_financial_data_file = pd.read_csv(path + "quarterly_financial_data_file_data_20260309_112648.csv") # Refinitiv Data

annual_financial_data_file = pd.read_csv(path + "annual_financial_data_file_data_20260309_115710.csv") # Refinitiv Data
annual_history_file = pd.read_csv(path + "annual_history_file_data_20260111_214138.csv") # Refinitiv Data

dividend_data_file = pd.read_csv(path + "dividend_data_file_data_20260309_105840.csv") # Refinitiv Data
once_data_file = pd.read_csv(path + "once_data_file_data_20260309_124826.csv") # Refinitiv Data

In [5]:
# Default Spread
bond_yields_AAA_file = pd.read_csv(path + "AAA.csv") #https://fred.stlouisfed.org/series/AAA
bond_yields_BAA_file = pd.read_csv(path + "BAA.csv") #https://fred.stlouisfed.org/series/BAA

# Term Spread
ten_year_gov_bond_yield_file = pd.read_csv(path + "ECB Data Portal_20260311082554.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y
three_month_gov_bond_yield_file = pd.read_csv(path + "ECB Data Portal_20260311082821.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3M

cepr_recession_indicator = pd.read_csv(path + "CEPR_Recession_Indicator.csv", sep = ";", decimal = ",") #https://eabcn.org/dbc/graphs

# Risk Free Rate for Beta
beta_0_file = pd.read_csv(path + "ECB Data Portal_20260311083245.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0
beta_1_file = pd.read_csv(path + "ECB Data Portal_20260311084919.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA1
beta_2_file = pd.read_csv(path + "ECB Data Portal_20260311084954.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA2
beta_3_file = pd.read_csv(path + "ECB Data Portal_20260311085031.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA3
tau_1_file = pd.read_csv(path + "ECB Data Portal_20260311085112.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.TAU1
tau_2_file = pd.read_csv(path + "ECB Data Portal_20260311085141.csv") #https://data.ecb.europa.eu/data/datasets/YC/YC.B.U2.EUR.4F.G_N_A.SV_C_YM.TAU2

fama_french_europe_5_factors = pd.read_csv(path + "Europe_5_Factors.csv", skiprows=6, index_col=0) #https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Europe_5_Factors_CSV.zip
fama_french_europe_momentum_factor = pd.read_csv(path + "Europe_MOM_Factor.csv", skiprows=6, index_col=0) #https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Europe_Mom_Factor_CSV.zip

In [ ]:
failed_files = {}

# Load all CSV files from the failed directory (Those are stocks for which the data loading process failed. They are stored for transparency)
for file in os.listdir(failed_path):
    if file.endswith(".csv"):
        failed_files[file] = pd.read_csv(failed_path + file)

In [7]:
print("Daily History File Columns:", daily_history_file.columns)
print("Daily Index History File Columns:", daily_index_history_file.columns)
print("Dividend Data File Columns:", dividend_data_file.columns)
print("Quarterly Financial Data File Columns:", quarterly_financial_data_file.columns)
print("Annual Financial Data File Columns:", annual_financial_data_file.columns)
print("Annual History File Columns:", annual_history_file.columns)
print("Once Data File Columns:", once_data_file.columns)

print("Bond Yields AAA File Columns:", bond_yields_AAA_file.columns)
print("Bond Yields BAA File Columns:", bond_yields_BAA_file.columns)

Daily History File Columns: Index(['Date', 'Price Close', 'Company Market Cap', 'Company Market Cap.1',
       'Company Market Cap.2', 'Outstanding Shares',
       'Price To Book Value Per Share (Daily Time Series Ratio)', 'Stock'],
      dtype='object')
Daily Index History File Columns: Index(['Date', 'Price Close', 'Currency', 'Stock'], dtype='object')
Dividend Data File Columns: Index(['Instrument', 'Dividend Ex Date', 'Period End Date',
       'Adjusted Gross Dividend Amount', 'Period End Date.1', 'Dividend Type',
       'Dividend Payment Type', 'Dividend Currency', 'Dividend Is Rescinded'],
      dtype='object')
Quarterly Financial Data File Columns: Index(['Instrument', 'Operating Income', 'Period End Date', 'Total Equity',
       'Period End Date.1', 'Total Assets, Reported', 'Period End Date.2',
       'Income Statement Orig Announce Date'],
      dtype='object')
Annual Financial Data File Columns: Index(['Instrument', 'Operating Income', 'Period End Date', 'Total Equity',
    

In [8]:
print("10 Year Government Bond Yield File Columns:", ten_year_gov_bond_yield_file.columns)
print("3 Month Government Bond Yield File Columns:", three_month_gov_bond_yield_file.columns)

print("CEPR Recession Indicator Columns:", cepr_recession_indicator.columns)

print("Beta 0 File Columns:", beta_0_file.columns)
print("Beta 1 File Columns:", beta_1_file.columns)
print("Beta 2 File Columns:", beta_2_file.columns)
print("Beta 3 File Columns:", beta_3_file.columns)
print("Tau 1 File Columns:", tau_1_file.columns)
print("Tau 2 File Columns:", tau_2_file.columns)

print("Fama French Research Data 5 Factors Columns:", fama_french_europe_5_factors.columns)
print("Fama French Research Data Momentum Factor Columns:", fama_french_europe_momentum_factor.columns)

10 Year Government Bond Yield File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - 10-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y)'],
      dtype='object')
3 Month Government Bond Yield File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - 3-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3M)'],
      dtype='object')
CEPR Recession Indicator Columns: Index(['Dates', 'Peak excluded', 'Peak included'], dtype='object')
Beta 0 File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - Beta 0 (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA0)'],
      dtype='object')
Beta 1 File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - Beta 1 (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA1)'],
      dtype='object')
Beta 2 File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - Beta 2 (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.BETA2)'],
      dtype='object')
Beta 3 File Columns: Index(['DATE', 'TIME PERIOD',
       'AAA yield curve - Beta 3 (YC.B.U2.EUR.4F.

In [9]:
for failed_file, df in failed_files.items():
    print(f"'{failed_file}' Columns:", df.columns)

'annual_history_file_failed_20260111_214138.csv' Columns: Index(['RIC'], dtype='object')


## 2. Prepare Dataframes for Transposing Columns

In this section, I prepare the dataframes for transposing the columns to wide format.

### 2.1 Daily History File

In [ ]:
daily_history_file_renamed = daily_history_file.rename(columns={"Date": "date"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Stock": "stock"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Price Close": "price_close_stocks"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Company Market Cap": "company_market_cap"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Company Market Cap.1": "free_float_mcap"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Company Market Cap.2": "outstanding_mcap"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Outstanding Shares": "outstanding_shares"})
daily_history_file_renamed = daily_history_file_renamed.rename(columns={"Price To Book Value Per Share (Daily Time Series Ratio)": "price_to_book_value_per_share"})

# Calculates book_to_price_value_per_share as the inverse of price_to_book_value_per_share
daily_history_file_renamed["book_to_price_value_per_share"] = 1 / daily_history_file_renamed["price_to_book_value_per_share"]

# Drops unnecessary columns
daily_history_file_renamed.drop(columns=["price_to_book_value_per_share"], inplace=True)

daily_history_file_renamed.head()

,date,price_close_stocks,company_market_cap,free_float_mcap,outstanding_mcap,outstanding_shares,stock,book_to_price_value_per_share
0,2015-10-06,26.500000,NaN,NaN,NaN,NaN,1COVG.DE^L25,NaN
1,2015-10-07,26.400000,5346000000.000000,1649999999.999990,5346000000.000000,202500000.000000,1COVG.DE^L25,NaN
2,2015-10-08,25.980000,5260950000.000000,1623749999.999990,5260950000.000000,202500000.000000,1COVG.DE^L25,NaN
3,2015-10-09,26.200000,5305500000.000000,1637500000.000000,5305500000.000000,202500000.000000,1COVG.DE^L25,NaN
4,2015-10-12,26.000000,5264999999.999990,1624999999.999990,5264999999.999990,202500000.000000,1COVG.DE^L25,NaN


### 2.2 Daily Index History File

In [11]:
# Renaming columns
daily_index_history_file_renamed = daily_index_history_file.rename(columns={"Date": "date"})
daily_index_history_file_renamed = daily_index_history_file_renamed.rename(columns={"Stock": "stock"})
daily_index_history_file_renamed = daily_index_history_file_renamed.rename(columns={"Price Close": "price_close_index"})

# Currency is not needed as all stocks are in the same currency (EUR)
daily_index_history_file_renamed.drop(columns=["Currency"], inplace=True)

daily_index_history_file_renamed.head()

,date,price_close_index,stock
0,2009-12-31,253.890000,.STOXX
1,2010-01-04,257.650000,.STOXX
2,2010-01-05,257.590000,.STOXX
3,2010-01-06,257.960000,.STOXX
4,2010-01-07,258.040000,.STOXX


### 2.3 Financial and Dividend Data Files

#### 2.3.1 Helper Functions

In [ ]:
def check_date_end_columns(file, file_name, date_columns = ["Period End Date", "Period End Date.1", "Period End Date.2"]):
    """
    This function checks if the specified date columns in the given DataFrame are consistent for each row.
    It prints out any rows where the dates are not consistent.
    """

    file_check = file.copy()

    # Creates a new column that checks if all date columns are the same for each row
    file_check["dates_consistent"] = file_check[date_columns].nunique(axis=1, dropna=True).le(1)

    # Filters rows where dates are not consistent
    bad_rows = file_check.loc[~file_check["dates_consistent"], ["Instrument"] + date_columns]

    if bad_rows.empty:
        print(f"Sanity Check [{file_name}]: All period end date columns of operating income, total equity and total assets are consistent.")
    else:
        print(f"Sanity Check [{file_name}]: Inconsistent date rows found:")
        print(bad_rows)

In [ ]:
def check_date_column_quality(file, file_name):
    """
    This function checks the quality of the "date" and "period_end_date" columns in the given DataFrame by ensuring that "period_end_date" is not greater than "date".
    """

    # Creates a mask for rows where period_end_date is greater than date
    bad_mask = file["date"].notna() & file["period_end_date"].notna() & (file["period_end_date"] > file["date"])
    n_bad = bad_mask.sum()

    if n_bad > 0:

        print("")
        print(f"Sanity Check [{file_name}]: There are {n_bad} rows where period_end_date is greater than date!")

        bad_rows = file.loc[bad_mask, ["stock", "date", "period_end_date"]]
        print(f"Sanity Check [{file_name}]: Rows where period_end_date is greater than date:")
        print(bad_rows)
    else:
        print("")
        print(f"Sanity Check [{file_name}]: All rows have period_end_date less than or equal to date. Date column quality check passed!")

In [ ]:
def rename_columns(file, rename_dict):
    """
    This function renames columns in the given DataFrame.
    """

    # Creates a copy of the original DataFrame to avoid modifying it directly
    file_renamed = file.copy()
    file_renamed = file_renamed.rename(columns=rename_dict)

    return file_renamed

In [ ]:
def remove_duplicates(file):
    """ 
    This function removes duplicate rows in the given DataFrame based on "stock" and "date" columns, keeping the latest entry based on "period_end_date". 
    """

    # Identify duplicate rows based on "stock" and "date"
    dup_mask = file.duplicated(subset=["stock", "date"], keep=False)
    dups = file.loc[dup_mask].sort_values(["stock", "date"])
    drop_rows = file.duplicated(subset=["stock", "date"], keep="last").sum()

    print("There are {} duplicate rows based on stock and date. Of those, {} will be dropped. The latest duplicates will be kept (Based on period_end_date).".format(dup_mask.sum(), drop_rows))

    # Removes duplicates, keeping the last occurrence based on "period_end_date"
    file = (
        file.sort_values(["stock", "date", "period_end_date"])
        .drop_duplicates(subset=["stock", "date"], keep="last")
        .reset_index(drop=True)
    )

    print("There are {} rows remaining after creating date column and removing duplicates.".format(len(file)))

    return file, dups

In [ ]:
def sum_duplicates(file):
    """
    This function sums the values of duplicate rows in the given DataFrame based on "stock" and "date" columns.
    It is used for dividend payments as they can have multiple entries for the same stock and date due to different dividend payment types 
    (e.g., cash, stock, special dividends).
    """

    # Identify duplicate rows based on "stock" and "date"
    dup_mask = file.duplicated(subset=["stock", "date"], keep=False)
    dups = file.loc[dup_mask].sort_values(["stock", "date"])

    # Sums duplicate rows
    sum_file = (
        file.groupby(["stock", "date"], as_index=False)
        .sum(numeric_only=True)
    )

    print("There are {} duplicate rows based on stock and date. These will be summed as they are the same dividend but represent different dividend payment types.".format(dup_mask.sum()))
    print("Out of {}, there are {} rows remaining after summing duplicates.".format(len(file), len(sum_file)))

    return sum_file, dups

In [ ]:
def create_date_column(file, file_name):
    """
    This function creates a unified date column in the given DataFrame by combining and processing various date columns.
    """
    
    print("There are {} rows before creating date column.".format(len(file)))

    # ---------------- Create date column ----------------

    # Creates a period_end_date column by backfilling the Period End Date columns of operating income, total equity, and total assets
    file["period_end_date"] = file[["Period End Date", "Period End Date.1", "Period End Date.2"]].bfill(axis=1).iloc[:, 0]   

    # Converts accounting_announce_date and period_end_date to datetime
    file["date"] = pd.to_datetime(file["accounting_announce_date"], errors="coerce")
    file["period_end_date"] = pd.to_datetime(file["period_end_date"], errors="coerce")

    # For rows where date is not NaT, set date to the end of the quarter
    file["date"] = file["date"].dt.to_period("Q").dt.end_time.dt.normalize()
    print("Number of rows with valid quarter dates from accounting_announce_date: {}".format(file["date"].notna().sum()))

    # Finds rows where date is NaT and period_end_date is not NaT
    mask = file["date"].isna() & file["period_end_date"].notna()
    print("Number of rows where accounting_announce_date is NaT but period_end_date is valid: {}".format(len(file[mask])))

    # For these rows, set date to period_end_date plus one quarter end
    file.loc[mask, "date"] = (file.loc[mask, "period_end_date"] + QuarterEnd(1))

    # Drops rows where date is still NaT
    print("There are {} rows with empty dates being dropped".format(file["date"].isna().sum()))
    file = file.dropna(subset=["date"])

    # ---------------- Remove duplicates based on stock and date ----------------

    # Checks for duplicates based on stock and date
    file, dups = remove_duplicates(file)

    # ---------------- Quality check date column ----------------
    
    check_date_column_quality(file, file_name)

    return file, dups

In [ ]:
def split_and_rename_financial_data(file, frequency_type):
    """
    This function splits and renames financial data columns in the given DataFrame based on the specified frequency type.
    """

    rename_dict = {
        "Instrument": "stock",
        "Operating Income": f"operating_income_{frequency_type}",
        "Total Equity": f"total_equity_{frequency_type}",
        "Total Assets": f"total_assets_{frequency_type}",
        "Total Assets, Reported": f"total_assets_reported_{frequency_type}",
        "Income Statement Orig Announce Date": "accounting_announce_date"
    }

    # Renames columns
    file_renamed = rename_columns(file, rename_dict)

    # Creates date column and remove duplicates for financial data 
    file_renamed, financial_dups = create_date_column(file_renamed, frequency_type)

    # Splits files based on financial metrics
    operating_income_file = file_renamed[["stock", f"operating_income_{frequency_type}", "date"]]
    total_equity_file = file_renamed[["stock", f"total_equity_{frequency_type}", "date"]]
    total_assets_reported_file = file_renamed[["stock", f"total_assets_reported_{frequency_type}", "date"]]
    
    return operating_income_file, total_equity_file, total_assets_reported_file, financial_dups

#### 2.3.2 Dividend Data Files

In [ ]:
dividend_file = dividend_data_file.copy()

rename_dict = {
    "Instrument": "stock",
    "Dividend Ex Date": "date",
    "Adjusted Gross Dividend Amount": "adjusted_gross_dividend_amount",
    "Dividend Payment Type": "dividend_payment_type",
    "Dividend Is Rescinded": "dividend_is_rescinded"
}

# Renames columns
dividend_file = rename_columns(dividend_file, rename_dict)

# Keeps only relevant columns and drop rows with missing dates
dividend_file = dividend_file[["stock", "date", "adjusted_gross_dividend_amount", "dividend_payment_type", "dividend_is_rescinded"]].reset_index(drop=True)
dividend_file = dividend_file.dropna(subset=["date"])
dividend_file["date"] = pd.to_datetime(dividend_file["date"], errors="coerce")

dropped_rows = dividend_file[dividend_file["dividend_payment_type"].isin(["Cash Dividend", "Cash and Stock Alternative", "Stock and Cash Alternative"]) == False]
print("There are {} rows with dividend payment types other than Cash Dividend, Cash and Stock Alternative, or Stock and Cash Alternative being dropped.".format(len(dropped_rows)))

# Keeps only rows with specified dividend payment types
dividend_file = dividend_file[dividend_file["dividend_payment_type"].isin(["Cash Dividend", "Cash and Stock Alternative", "Stock and Cash Alternative"])]

dropped_rows = dividend_file[dividend_file["dividend_is_rescinded"] == True]
print("There are {} rows with cancelled dividends being dropped.".format(len(dropped_rows)))

# Removes rows where dividend is rescinded
dividend_file = dividend_file[dividend_file["dividend_is_rescinded"] != "True"]

dividend_file = dividend_file[["stock", "date", "adjusted_gross_dividend_amount"]]

dividend_file, dividend_dups = sum_duplicates(dividend_file)

There are 99 rows with dividend payment types other than Cash Dividend, Cash and Stock Alternative, or Stock and Cash Alternative being dropped.
There are 70 rows with cancelled dividends being dropped.
There are 1986 duplicate rows based on stock and date. These will be summed as they are the same dividend but represent different dividend payment types.
Out of 18008, there are 17006 rows remaining after summing duplicates.


In [20]:
dividend_file.head()

,stock,date,adjusted_gross_dividend_amount
0,1COVG.DE^L25,2016-05-04,0.700000
1,1COVG.DE^L25,2017-05-04,1.350000
2,1COVG.DE^L25,2018-04-16,2.200000
3,1COVG.DE^L25,2019-04-15,2.400000
4,1COVG.DE^L25,2020-07-31,1.200000


#### 2.3.4 Quarterly Financial Data Files

In [21]:
check_date_end_columns(quarterly_financial_data_file, "quarterly")

Sanity Check [quarterly]: All period end date columns of operating income, total equity and total assets are consistent.


In [22]:
quarterly_financial_data_file_renamed = quarterly_financial_data_file.copy()

quarterly_operating_income_file, quarterly_total_equity_file, quarterly_total_assets_reported_file, quarterly_financial_dups = split_and_rename_financial_data(quarterly_financial_data_file_renamed, "quarterly")

There are 41797 rows before creating date column.
Number of rows with valid quarter dates from accounting_announce_date: 31081
Number of rows where accounting_announce_date is NaT but period_end_date is valid: 150
There are 10566 rows with empty dates being dropped
There are 1123 duplicate rows based on stock and date. Of those, 619 will be dropped. The latest duplicates will be kept (Based on period_end_date).
There are 30612 rows remaining after creating date column and removing duplicates.

Sanity Check [quarterly]: All rows have period_end_date less than or equal to date. Date column quality check passed!


#### 2.3.5 Annual Financial Data Files

In [23]:
check_date_end_columns(annual_financial_data_file, "annual")

Sanity Check [annual]: All period end date columns of operating income, total equity and total assets are consistent.


In [24]:
annual_financial_data_file_renamed = annual_financial_data_file.copy()

annual_operating_income_file, annual_total_equity_file, annual_total_assets_reported_file, annual_financial_dups = split_and_rename_financial_data(annual_financial_data_file_renamed, "annual")

There are 14481 rows before creating date column.
Number of rows with valid quarter dates from accounting_announce_date: 14425
Number of rows where accounting_announce_date is NaT but period_end_date is valid: 2
There are 54 rows with empty dates being dropped
There are 502 duplicate rows based on stock and date. Of those, 334 will be dropped. The latest duplicates will be kept (Based on period_end_date).
There are 14093 rows remaining after creating date column and removing duplicates.

Sanity Check [annual]: All rows have period_end_date less than or equal to date. Date column quality check passed!


#### 2.3.6 Merging Data

In [ ]:
def add_missing_quarters(df, column_name, frequency_type, sum = False):
    """
    This function adds missing quarters to the given DataFrame and calculates the trailing twelve months (TTM) sum for a specified column.
    """

    # Creates a "q" column representing the quarter period
    df["q"] = df["date"].dt.to_period("Q")

    # Sets "q" as the index after sorting by "q"
    df = df.sort_values("q").set_index("q")

    # Creates a complete range of quarters from the minimum to maximum quarter in the data
    full_q = pd.period_range(df.index.min(), df.index.max(), freq="Q")
    df = df.reindex(full_q)  # missing quarters become rows with NaN

    # Makes sure date and stock columns are filled correctly
    df["date"] = df.index.to_timestamp(how="end").normalize()
    df["stock"] = df["stock"].iloc[0]

    # Calculates TTM sum for the specified column if sum = True
    if sum:
        df[f"{column_name}_quarterly_ttm"] = df[f"{column_name}_{frequency_type}"].rolling(4, min_periods=4).sum()
    else:
        df[f"{column_name}_quarterly_ttm"] = df[f"{column_name}_{frequency_type}"]

    # Fills TTM values with annual data where quarterly TTM is not available
    df[f"{column_name}_ttm"] = df[f"{column_name}_quarterly_ttm"].fillna(df[f"{column_name}_annual"])

    # Keeps only relevant columns
    df = df[["stock", "date", f"{column_name}_ttm"]]

    return df

In [ ]:
def merge_data(base_file, merge_file, value_column, sum = False):
    """
    This function merges two DataFrames on "stock" and "date", sorts the result, and adds missing quarters with 
    TTM calculations for a specified value column.
    """

    df = base_file.merge(merge_file, on=["stock", "date"], how="outer")

    df = df.sort_values(["stock", "date"])

    # Adds missing quarters and calculate TTM
    ttm = (
        df.groupby("stock", group_keys=False)
            .apply(add_missing_quarters, column_name=value_column, frequency_type="quarterly", sum = sum)
            .reset_index(drop=True)
    )

    return ttm

In [27]:
operating_income_ttm = merge_data(quarterly_operating_income_file, annual_operating_income_file, "operating_income", sum = True)
total_equity_ttm = merge_data(quarterly_total_equity_file, annual_total_equity_file, "total_equity", sum = False)
total_assets_reported_ttm = merge_data(quarterly_total_assets_reported_file, annual_total_assets_reported_file, "total_assets_reported", sum = False)

/tmp/ipykernel_3838/501007137.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_missing_quarters, column_name=value_column, frequency_type="quarterly", sum = sum)
/tmp/ipykernel_3838/501007137.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_missing_quarters, column_name=value_column, frequency_type="quarterly", sum = sum)
/tmp/ipykernel_3838/501007137.py:14: FutureWarning: DataFrameGroup

In [28]:
operating_income_ttm.head()

,stock,date,operating_income_ttm
0,1COVG.DE^L25,2014-12-31,NaN
1,1COVG.DE^L25,2015-03-31,NaN
2,1COVG.DE^L25,2015-06-30,NaN
3,1COVG.DE^L25,2015-09-30,NaN
4,1COVG.DE^L25,2015-12-31,NaN


In [29]:
total_equity_ttm.head()

,stock,date,total_equity_ttm
0,1COVG.DE^L25,2014-12-31,2854000000.000000
1,1COVG.DE^L25,2015-03-31,NaN
2,1COVG.DE^L25,2015-06-30,NaN
3,1COVG.DE^L25,2015-09-30,NaN
4,1COVG.DE^L25,2015-12-31,1104000000.000000


In [30]:
total_assets_reported_ttm.head()

,stock,date,total_assets_reported_ttm
0,1COVG.DE^L25,2014-12-31,11364000000.000000
1,1COVG.DE^L25,2015-03-31,NaN
2,1COVG.DE^L25,2015-06-30,NaN
3,1COVG.DE^L25,2015-09-30,NaN
4,1COVG.DE^L25,2015-12-31,10825000000.000000


### 2.4 Annual History File

In [31]:
annual_history_file_renamed = annual_history_file.rename(columns={"Date": "date"})
annual_history_file_renamed = annual_history_file_renamed.rename(columns={"ESG Score": "esg_score"})
annual_history_file_renamed = annual_history_file_renamed.rename(columns={"Stock": "stock"})

annual_history_file_renamed.head()

,date,esg_score,stock
0,12/31/2010,NaN,1COVG.DE^L25
1,12/31/2011,NaN,1COVG.DE^L25
2,12/31/2012,NaN,1COVG.DE^L25
3,12/31/2013,NaN,1COVG.DE^L25
4,12/31/2014,NaN,1COVG.DE^L25


### 2.5 Once Data File

In [32]:
once_data_file.head()

,Instrument,GICS Sector Name,ICB Sector name,ICB Industry name,Currency,Country of Exchange,Primary Country of Risk,Country of Headquarters
0,1COVG.DE^L25,Materials,Chemicals,Basic Materials,EUR,Germany,Germany,Germany
1,1COVG.DE^L25,NaN,NaN,NaN,EUR,NaN,NaN,NaN
2,1COVG.DE^L25,NaN,NaN,NaN,EUR,NaN,NaN,NaN
3,1COVG.DE^L25,NaN,NaN,NaN,EUR,NaN,NaN,NaN
4,1COVG.DE^L25,NaN,NaN,NaN,EUR,NaN,NaN,NaN


In [ ]:
once_data_file_renamed = once_data_file.rename(columns={
    "Instrument": "stock",
    "ICB Industry name": "industry",
    "Currency": "currency",
    "Country of Headquarters": "country_of_headquarters",
    })

print("There are " + str(len(once_data_file_renamed)) + " rows in total!")

# Drops rows where every cell of industry and country_of_headquarters is missing
once_data_file_renamed = once_data_file_renamed.dropna(subset=["industry", "country_of_headquarters"], how="all")

print("There are " + str(len(once_data_file_renamed)) + " rows remaining after dropping rows with missing values!")

industry_df = once_data_file_renamed[["stock", "industry"]].reset_index(drop=True)
currency_df = once_data_file_renamed[["stock", "currency"]].reset_index(drop=True)
country_of_headquarters_df = once_data_file_renamed[["stock", "country_of_headquarters"]].reset_index(drop=True)


There are 3962588 rows in total!
There are 967 rows remaining after dropping rows with missing values!


In [35]:
industry_df.head()

,stock,industry
0,1COVG.DE^L25,Basic Materials
1,1U1.DE,Telecommunications
2,A2.MI,Utilities
3,AAAA.L^C21,Consumer Discretionary
4,AAF.L,Telecommunications


In [36]:
currency_df.head()

,stock,currency
0,1COVG.DE^L25,EUR
1,1U1.DE,EUR
2,A2.MI,EUR
3,AAAA.L^C21,GBp
4,AAF.L,GBp


In [37]:
country_of_headquarters_df.head()

,stock,country_of_headquarters
0,1COVG.DE^L25,Germany
1,1U1.DE,Germany
2,A2.MI,Italy
3,AAAA.L^C21,United Kingdom
4,AAF.L,United Kingdom


### 2.6 FF-5 Factors

In [38]:
# Creates a copy of the original Fama-French Europe 5 Factors DataFrame to avoid modifying it directly
ff5 =  fama_french_europe_5_factors.copy()

# Converts the index to datetime format and adjust to the end of the month
ff5.index = pd.to_datetime(ff5.index, format='%Y%m') + MonthEnd(0)
ff5.index = ff5.index.date
ff5.index.name = "date"

# Converts factors to decimals
columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]

for col in columns:
    ff5[col] = ff5[col] / 100.0

# Resets index to date column
ff5 = ff5.reset_index()

ff5.head()

,date,Mkt-RF,SMB,HML,RMW,CMA,RF
0,1990-07-31,0.044600,0.002000,-0.015200,0.002800,0.011000,0.006800
1,1990-08-31,-0.108800,0.002500,-0.003000,-0.005000,0.016600,0.006600
2,1990-09-30,-0.121900,0.019200,0.004400,0.003300,0.017800,0.006000
3,1990-10-31,0.064500,-0.025800,-0.011200,0.008600,-0.007300,0.006800
4,1990-11-30,-0.004200,-0.026600,0.005700,0.002500,-0.003300,0.005700


In [39]:
# Creates a copy of the original Fama-French Europe Momentum Factor DataFrame to avoid modifying it directly
ff_mom = fama_french_europe_momentum_factor.copy()

# Converts the index to datetime format and adjust to the end of the month
ff_mom.index = pd.to_datetime(ff_mom.index, format='%Y%m') + MonthEnd(0)
ff_mom.index = ff_mom.index.date
ff_mom.index.name = "date"

ff_mom["WML"] = ff_mom["WML"] / 100.0

# Resets index to date column
ff_mom = ff_mom.reset_index()

ff_mom.head()

,date,WML
0,1990-11-30,0.024500
1,1990-12-31,0.005200
2,1991-01-31,-0.012500
3,1991-02-28,-0.075000
4,1991-03-31,0.002300


In [40]:
# Merges the Fama-French 5 Factors and Momentum Factor data on the date column, keeping all dates from both datasets
fama_french_data = ff5.merge(ff_mom, on="date", how="outer")

fama_french_data.head()

,date,Mkt-RF,SMB,HML,RMW,CMA,RF,WML
0,1990-07-31,0.044600,0.002000,-0.015200,0.002800,0.011000,0.006800,NaN
1,1990-08-31,-0.108800,0.002500,-0.003000,-0.005000,0.016600,0.006600,NaN
2,1990-09-30,-0.121900,0.019200,0.004400,0.003300,0.017800,0.006000,NaN
3,1990-10-31,0.064500,-0.025800,-0.011200,0.008600,-0.007300,0.006800,NaN
4,1990-11-30,-0.004200,-0.026600,0.005700,0.002500,-0.003300,0.005700,0.024500


### 2.7 Constituents Presence Matrix

In [41]:
constituents_presence_matrix = constituents_presence_matrix_file.copy()

# Renames "Date" column to "date"
constituents_presence_matrix = constituents_presence_matrix.rename(columns={"Date": "date"})

### 2.8 Moody's Seasoned Corporate Bond Yield (AAA & BAA)

In [42]:
bond_yields_AAA = bond_yields_AAA_file.copy()
bond_yields_BAA = bond_yields_BAA_file.copy()

# Renames observation_date column to date
bond_yields_AAA = bond_yields_AAA.rename(columns={"observation_date": "date"})
bond_yields_BAA = bond_yields_BAA.rename(columns={"observation_date": "date"})

# Converts date columns to datetime format
bond_yields_AAA["date"] = pd.to_datetime(bond_yields_AAA["date"], errors="coerce").dt.date
bond_yields_BAA["date"] = pd.to_datetime(bond_yields_BAA["date"], errors="coerce").dt.date

# Merges AAA and BAA bond yields on date
bond_yields = bond_yields_AAA.merge(bond_yields_BAA, on="date", how="outer")

bond_yields["AAA"] = bond_yields["AAA"] / 100.0
bond_yields["BAA"] = bond_yields["BAA"] / 100.0

In [43]:
bond_yields.head()

,date,AAA,BAA
0,1919-01-01,0.053500,0.071200
1,1919-02-01,0.053500,0.072000
2,1919-03-01,0.053900,0.071500
3,1919-04-01,0.054400,0.072300
4,1919-05-01,0.053900,0.070900


### 2.9 Ten Year & Three Month Government Bond Yield

In [44]:
ten_year_gov_bond_yield = ten_year_gov_bond_yield_file.copy()
three_month_gov_bond_yield = three_month_gov_bond_yield_file.copy()

# Renames columns for clarity
ten_year_gov_bond_yield = ten_year_gov_bond_yield.rename(columns={"DATE": "date", 
                                                                  "AAA yield curve - 10-year spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y)": "spot_rate_10y_government_bond"})

three_month_gov_bond_yield = three_month_gov_bond_yield.rename(columns={"DATE": "date",
                                                                  "AAA yield curve - 3-month spot rate (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3M)": "spot_rate_3m_government_bond"})

# Converts date column to datetime format
ten_year_gov_bond_yield["date"] = pd.to_datetime(ten_year_gov_bond_yield["date"], errors="coerce").dt.date
three_month_gov_bond_yield["date"] = pd.to_datetime(three_month_gov_bond_yield["date"], errors="coerce").dt.date

ten_year_gov_bond_yield["spot_rate_10y_government_bond"] = ten_year_gov_bond_yield["spot_rate_10y_government_bond"] / 100.0
three_month_gov_bond_yield["spot_rate_3m_government_bond"] = three_month_gov_bond_yield["spot_rate_3m_government_bond"] / 100.0

# Drops unnecessary TIME PERIOD column
ten_year_gov_bond_yield = ten_year_gov_bond_yield.drop(columns=["TIME PERIOD"])
three_month_gov_bond_yield = three_month_gov_bond_yield.drop(columns=["TIME PERIOD"])

# Merges ten-year and three-month government bond yields on date
gov_bond_yields = ten_year_gov_bond_yield.merge(three_month_gov_bond_yield, on="date", how="outer")


In [45]:
gov_bond_yields.head()

,date,spot_rate_10y_government_bond,spot_rate_3m_government_bond
0,2004-09-06,0.042092,0.020342
1,2004-09-07,0.042096,0.020409
2,2004-09-08,0.042284,0.020444
3,2004-09-09,0.041619,0.020371
4,2004-09-10,0.041210,0.020346


### 2.10 Svensson Parameters

In [46]:
def get_svensson_column_name(parameter_1, parameter_2):

    return f"AAA yield curve - {parameter_1} (YC.B.U2.EUR.4F.G_N_A.SV_C_YM.{parameter_2})"

In [47]:
# Dictionary mapping beta and tau files to their respective parameter names

beta_0_renamed = beta_0_file.copy()
beta_1_renamed = beta_1_file.copy()
beta_2_renamed = beta_2_file.copy()
beta_3_renamed = beta_3_file.copy()
tau_1_renamed = tau_1_file.copy()
tau_2_renamed = tau_2_file.copy()

files = {
    ("Beta 0", "BETA0"): beta_0_renamed,
    ("Beta 1", "BETA1"): beta_1_renamed,
    ("Beta 2", "BETA2"): beta_2_renamed,
    ("Beta 3", "BETA3"): beta_3_renamed,
    ("Tau 1", "TAU1"): tau_1_renamed,
    ("Tau 2", "TAU2"): tau_2_renamed
}

# Rename columns in beta and tau files
for (param_0, param_1), file in files.items():

    # Drop unnecessary TIME PERIOD column
    file.drop(columns=["TIME PERIOD"], inplace=True)

    file.rename(
        columns={
            "DATE": "date",
            get_svensson_column_name(param_0, param_1): param_1.lower()
        } 
        , inplace=True)
    
    # Convert date column to datetime format
    file["date"] = pd.to_datetime(file["date"], errors="coerce").dt.date

merged_svensson_parameters = beta_0_renamed.merge(beta_1_renamed, on="date", how="outer") \
    .merge(beta_2_renamed, on="date", how="outer") \
    .merge(beta_3_renamed, on="date", how="outer") \
    .merge(tau_1_renamed, on="date", how="outer") \
    .merge(tau_2_renamed, on="date", how="outer")

merged_svensson_parameters.head()

,date,beta0,beta1,beta2,beta3,tau1,tau2
0,2004-09-06,5.410510,-3.462358,-0.361335,-0.466368,3.128331,1.489535
1,2004-09-07,5.391886,-3.450353,-0.372908,-0.271295,3.157868,1.548566
2,2004-09-08,5.385978,-3.447950,-0.346505,-0.198077,3.137857,1.553148
3,2004-09-09,5.377333,-3.432592,-0.382208,-0.293482,3.271392,1.521143
4,2004-09-10,5.355732,-3.395283,-0.375382,-0.507894,3.279876,1.479197


### 2.11 Risk Free Rate

In [48]:
def calculate_risk_free_rate(row, maturity):
    """
    This function calculates the risk-free rate using the Svensson model parameters.
    """

    beta_0 = row["beta0"]
    beta_1 = row["beta1"]
    beta_2 = row["beta2"]
    beta_3 = row["beta3"]
    tau_1 = row["tau1"]
    tau_2 = row["tau2"]

    t = maturity

    # Calculate the svensson parameters
    x1 = t / tau_1
    x2 = t / tau_2
    term1 = (1 - np.exp(-x1)) / x1
    term2 = term1 - np.exp(-x1)
    term3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)

    # Calculate annualized risk-free rate
    risk_free_rate = beta_0 + beta_1 * term1 + beta_2 * term2 + beta_3 * term3

    # Convert percentage to decimal
    risk_free_rate = risk_free_rate / 100.0 

    return risk_free_rate

In [49]:
risk_free_rate = merged_svensson_parameters.copy()

# Calculate the 1 month risk-free rate
risk_free_rate["risk_free_rate_annualized"] = risk_free_rate.apply(lambda row: calculate_risk_free_rate(row, maturity=1/12), axis=1)

# Convert annualized risk-free rate to monthly rate
risk_free_rate["risk_free_rate"] = np.exp((risk_free_rate["risk_free_rate_annualized"]) / 12) - 1

# Keep only relevant columns
risk_free_rate = risk_free_rate[["date", "risk_free_rate"]]

risk_free_rate.head()

,date,risk_free_rate
0,2004-09-06,0.001648
1,2004-09-07,0.001647
2,2004-09-08,0.001646
3,2004-09-09,0.001648
4,2004-09-10,0.001655


### 2.12 CEPR Recession Indicator

In [50]:
def map_quarter_to_month(dataframe, date_column):
    """
    Mapping quarter indicators in date column to months:

    Input Data:

    Dates       Indicator
    2020,00     1
    2020,25     1
    2020,50     0
    2020,75     0
    2021,00     1
    ...


    Output Data

    Dates           Indicator

    2020-01-31      1
    2020-02-29      1
    2020-03-31      1
    2020-04-30      1
    2020-05-31      1
    2020-06-30      1
    2020-07-31      0
    2020-08-31      0
    2020-09-30      0
    2020-10-31      1
    2020-11-30      1
    2020-12-31      1
    2021-01-31      1
    ...
    
    """

    df = dataframe.copy()

    # Converts values like "2020,25" to numeric year-quarter fractions
    date_as_float = pd.to_numeric(
        df[date_column].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Splits into year and quarter fraction
    df["_year"] = np.floor(date_as_float).astype(int)
    df["_quarter_fraction"] = (date_as_float - df["_year"]).round(2)

    # Maps quarter fractions to starting month of the quarter
    quarter_to_start_month = {0.00: 1, 0.25: 4, 0.50: 7, 0.75: 10}
    df["_start_month"] = df["_quarter_fraction"].map(quarter_to_start_month)

    # Converts to datetime format representing the start of the quarter
    df["_start_month"] = df["_start_month"].astype(int)
    df["_quarter_start"] = pd.to_datetime(
        {
            "year": df["_year"],
            "month": df["_start_month"],
            "day": 1
        },
        errors="coerce"
    )

    # Expands each quarter row to three monthly rows
    expanded_df = df.loc[df.index.repeat(3)].copy()
    expanded_df["_month_offset"] = expanded_df.groupby(level=0).cumcount()

    # Calculates the month for each expanded row by adding the month offset to the quarter start month
    month_index = expanded_df["_quarter_start"].dt.month - 1 + expanded_df["_month_offset"]
    expanded_year = expanded_df["_quarter_start"].dt.year + (month_index // 12)
    expanded_month = (month_index % 12) + 1

    # Creates a new date column representing the end of each month within the quarter
    expanded_df["date"] = pd.to_datetime(
        {"year": expanded_year, "month": expanded_month, "day": 1},
        errors="coerce"
    ) + MonthEnd(0)

    # Removes timestamp
    expanded_df["date"] = pd.to_datetime(expanded_df["date"], errors="coerce").dt.date

    # Final cleanup
    expanded_df = expanded_df.drop(
        columns=["_year", "_quarter_fraction", "_start_month", "_quarter_start", "_month_offset"]
    )

    # If the original date column is not "date", drop it after creating the new "date" column
    if date_column != "date":
        expanded_df = expanded_df.drop(columns=[date_column])

    # Sort by the new date column and reset index
    expanded_df = expanded_df.sort_values("date").reset_index(drop=True)

    # Rename columns for clarity
    expanded_df.rename(columns={"Peak excluded": "peak_excluded",
                                "Peak included": "peak_included"}, inplace=True)

    # Reorder columns to have date first, followed by peak_excluded and peak_included
    expanded_df = expanded_df[["date", "peak_excluded", "peak_included"]]

    return expanded_df

In [51]:
cepr_recession_indicator_renamed = cepr_recession_indicator.copy()

# Mapping data from "YYYY,00" / "YYYY,25" / "YYYY,50" / "YYYY,75" format to monthly date format
cepr_recession_indicator_renamed = map_quarter_to_month(cepr_recession_indicator_renamed, "Dates")

cepr_recession_indicator_renamed.head()

,date,peak_excluded,peak_included
0,1970-01-31,0,0
1,1970-02-28,0,0
2,1970-03-31,0,0
3,1970-04-30,0,0
4,1970-05-31,0,0


## 3. Transpose Data

Now I transpose the data

In [52]:
def getPreparedTimeSeriesDataForXLSXExport(time_series_data, value_column):

    # Subset data to only relevant columns and clean it
    data_subsetted = time_series_data[["date", "stock", value_column]]
    data_subsetted = data_subsetted.dropna()
    data_subsetted = data_subsetted[data_subsetted[value_column] != ""]
    data_subsetted = data_subsetted[data_subsetted[value_column] != "NaN"]

    # Convert Date column to datetime and sort values
    data_subsetted["date"] = pd.to_datetime(data_subsetted["date"], errors="coerce").dt.date
    data_subsetted = data_subsetted.sort_values(["date", "stock"], kind="mergesort")

    # Check for duplicates based on date and stock
    duplicates = data_subsetted.duplicated(subset=["date", "stock"])
    duplicated_rows = None
    
    # Remove duplicates if any exist
    if duplicates.any():
        print(f"The field {value_column} contained {duplicates.sum()} duplicate values. They will be removed from the data!")

        # Get all duplicated rows for reporting and export
        duplicates_all = data_subsetted.duplicated(subset=["date", "stock"], keep=False)
        duplicated_rows = data_subsetted[duplicates_all]

        # Remove duplicates, keeping the first occurrence
        data_subsetted = data_subsetted.drop_duplicates(subset=["date", "stock"])

    # Transpose data to have dates as rows and stocks as columns  
    transposed_data = data_subsetted.pivot(index="date", columns="stock", values=value_column)

    print(f"Successfully transposed {value_column}")
    
    return duplicates, duplicated_rows, transposed_data

In [53]:
# Gets column lists for each dataset
daily_history_file_renamed_columns = daily_history_file_renamed.columns.tolist()
daily_index_history_file_renamed_columns = daily_index_history_file_renamed.columns.tolist()
dividend_file_columns = dividend_file.columns.tolist()
operating_income_ttm_columns = operating_income_ttm.columns.tolist()
total_equity_ttm_columns = total_equity_ttm.columns.tolist()
total_assets_reported_ttm_columns = total_assets_reported_ttm.columns.tolist()
annual_history_file_renamed_columns = annual_history_file_renamed.columns.tolist()

# Creates dictionaries to hold column names and data for easy access
data_columns = {
    0: daily_history_file_renamed_columns,
    1: daily_index_history_file_renamed_columns,
    2: dividend_file_columns,
    3: operating_income_ttm_columns,
    4: total_equity_ttm_columns,
    5: total_assets_reported_ttm_columns,
    6: annual_history_file_renamed_columns
}

# Creates a list of dataframes for easy access
data = [
    daily_history_file_renamed,
    daily_index_history_file_renamed,
    dividend_file,
    operating_income_ttm,
    total_equity_ttm,
    total_assets_reported_ttm,
    annual_history_file_renamed
]

In [ ]:
transposed_dfs = {}
duplicates_dict = {}

for current_data_index, columns in data_columns.items():
    current_data = data[current_data_index]
    for col in columns:
        if col in ("date", "stock"):
            continue
        
        # Gets prepared time series data for XLSX export
        duplicates, duplicates_rows, transposed_df = getPreparedTimeSeriesDataForXLSXExport(current_data, col)

        # Ensures working sheet names for Excel
        safe_name_df = col[:31] if len(col) > 31 else col
        safe_name_dups = col[:26] + "_dups" if len(col) > 27 else col + "_dups"

        # Stores transposed dataframe
        transposed_dfs[f"{safe_name_df}"] = transposed_df

        # Stores duplicates if any
        if duplicates.any():
            duplicates_dict[f"{safe_name_dups}"] = duplicates_rows

The field price_close_stocks contained 1080 duplicate values. They will be removed from the data!
Successfully transposed price_close_stocks
The field company_market_cap contained 794 duplicate values. They will be removed from the data!
Successfully transposed company_market_cap
The field free_float_mcap contained 789 duplicate values. They will be removed from the data!
Successfully transposed free_float_mcap
The field outstanding_mcap contained 791 duplicate values. They will be removed from the data!
Successfully transposed outstanding_mcap
The field outstanding_shares contained 830 duplicate values. They will be removed from the data!
Successfully transposed outstanding_shares
The field book_to_price_value_per_share contained 1813 duplicate values. They will be removed from the data!
Successfully transposed book_to_price_value_per_share
Successfully transposed price_close_index
Successfully transposed adjusted_gross_dividend_amount
Successfully transposed operating_income_ttm
Succ

## 4. Export as XLSX File

Here I export the tranposed data to an xlsx file. 

In [ ]:
data_output_path = "data/processed_data/01_prepared_raw_data.xlsx"

other_files = {
    "industry": industry_df,
    "currency": currency_df,
    "country_of_headquarters": country_of_headquarters_df,
    "svensson_parameters":  merged_svensson_parameters,
    "risk_free_rate": risk_free_rate,
    "fama_french_data": fama_french_data,
    "bond_yields": bond_yields,
    "gov_bond_yields": gov_bond_yields,
    "recession_indicator": cepr_recession_indicator_renamed
}

duplicates_dict = {}

with pd.ExcelWriter(data_output_path, engine="xlsxwriter") as writer:

    print("Starting to write data to Excel sheets...")
    print("")

    # Writes constituents data to Excel sheets
    constituents_name_list_file.to_excel(writer, sheet_name="constituents", index = False)
    print("Written data sheet: constituents")
    constituents_presence_matrix.to_excel(writer, sheet_name="presence_matrix", index = False)
    print("Written data sheet: presence_matrix")
    
    # Writes transposed dataframes to Excel sheets
    for sheet_name, df in transposed_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name)
        print(f"Written data sheet: {sheet_name}")
        
    # Writes other dataframes to Excel sheets
    for sheet_name, df in other_files.items():

        # Ensure safe sheet names for Excel
        safe_name = sheet_name[:31] if len(sheet_name) > 31 else sheet_name

        df.to_excel(writer, sheet_name=safe_name, index = False)
        print(f"Written data sheet: {safe_name}")
    
print("")
print(f"All data successfully written to {data_output_path}")


Starting to write data to Excel sheets...

Written data sheet: constituents
Written data sheet: presence_matrix
Written data sheet: price_close_stocks
Written data sheet: company_market_cap
Written data sheet: free_float_mcap
Written data sheet: outstanding_mcap
Written data sheet: outstanding_shares
Written data sheet: book_to_price_value_per_share
Written data sheet: price_close_index
Written data sheet: adjusted_gross_dividend_amount
Written data sheet: operating_income_ttm
Written data sheet: total_equity_ttm
Written data sheet: total_assets_reported_ttm
Written data sheet: esg_score
Written data sheet: industry
Written data sheet: currency
Written data sheet: country_of_headquarters
Written data sheet: svensson_parameters
Written data sheet: risk_free_rate
Written data sheet: fama_french_data
Written data sheet: bond_yields
Written data sheet: gov_bond_yields
Written data sheet: recession_indicator

All data successfully written to data/processed_data/01_prepared_raw_data.xlsx


In [ ]:
error_output_path = "data/processed_data/01_issue_variables.xlsx"

with pd.ExcelWriter(error_output_path) as writer:

    print("Starting to write error data to Excel sheets...")
    print("")

    # Write duplicates information to Excel sheets for visual inspection
    for sheet_name, duplicate_dfs in duplicates_dict.items():
        if not duplicate_dfs.empty:
            duplicate_dfs.to_excel(writer, sheet_name=sheet_name, index = False)
            print(f"Written duplicates sheet: {sheet_name}")
    
    # Write failed files data to Excel sheets
    for failed_file, df in failed_files.items():
        safe_name = failed_file[:31] if len(failed_file) > 31 else failed_file.replace(".csv", "")
        df.to_excel(writer, sheet_name=safe_name, index = False)
        print(f"Written failed file sheet: {safe_name}")

print("")
print(f"All error data successfully written to {error_output_path}")

Starting to write error data to Excel sheets...

Written failed file sheet: annual_history_file_failed_2026

All error data successfully written to data/processed_data/issue_variables.xlsx
